In [4]:
import numpy as np

In [5]:
#Activation function class:
class ActivationFunction:
    def __init__(self):
        return
    def forward_pass(self,x):
        return
    def backward_pass(self, derivative):
        return

In [6]:
#Class for individual neuron (Relu)
class Relu(ActivationFunction):
    def __init__(self):
        super().__init__()
        self.x = None
        self.gradient = None

    def forward_pass(self,x):
        self.gradient = np.where(x > 0, 1, 0)
        self.x = x
        return self.x * self.gradient
    def backward_pass(self, derivative):
        return derivative * self.gradient

In [46]:
#Class for leaky ReLU
class LeakyRelu(ActivationFunction):
    def __init__(self, alpha=0.01):
        super().__init__()
        self.x = None
        self.gradient = None
        self.alpha = alpha
    def forward_pass(self,x):
        self.gradient = np.where(x > 0, 1, self.alpha)
        self.x = x
        return self.x * self.gradient
    def backward_pass(self, derivative):
        print(x)
        return derivative * self.gradient

In [62]:
#Parent class for neurons, by default ill keep this linear
class Layer:
    def __init__(self,input_dim,output_dim,learning_rate=0.001,activation_function=None,weights=None,biases=None):
        self.learning_rate = learning_rate
        self.x = None
        if weights is None:
            weights = np.random.uniform(-1,1,size=(output_dim,input_dim))
        if biases is None:
            biases = np.random.uniform(-1,1,size=(output_dim,1))
        if activation_function is None:
            activation_function = Relu()

        self.activation_function = activation_function
        self.weights = weights
        self.biases = biases
        self.input_dim = input_dim
        self.output_dim = output_dim
        return

    def forward_pass(self,x):
        self.x = x
        linear_pass = self.weights @ x + self.biases
        return self.activation_function.forward_pass(linear_pass)

    def backward_pass(self, derivative):
        activation_pass = self.activation_function.backward_pass(derivative)
        return self.weights.T @ activation_pass

    #Check validity of this code:
    def update(self, derivative):
        gradient = derivative * self.activation_function.gradient

        weight_gradient = gradient @ self.x.T
        bias_gradient = np.mean(gradient, axis=1,keepdims=True)

        self.weights -= self.learning_rate * weight_gradient
        self.biases -= self.learning_rate * bias_gradient

In [69]:
#Class to house the whole model, will contain multiple layers
class Model:
    def __init__(self, layers):
        self.layers = layers

    def forward_pass(self,x):
        input = x
        for i in self.layers:
            input = i.forward_pass(input)
        return input

    def backward_pass(self, derivative):
        gradient = derivative
        for i in reversed(self.layers):
            temp2 = i.backward_pass(gradient)
            i.update(gradient)
            gradient = temp2

    def train(self,x,y,iterations=10000,cutoff = 0.001):
        cost_history = []
        for i in range(iterations):
            prediction = self.forward_pass(x)
            loss = prediction - y
            cost_history.append(np.average(loss**2)/2)
            self.backward_pass(loss)
            if len(cost_history) < 2:
                continue
            if abs((cost_history[-1]-cost_history[-2])/cost_history[-1]) < cutoff:
                break
        return cost_history

    def train_advanced(self, x, y, epoch, batch_size, iterations=10000, cutoff = 0.001):
        cost_history = []
        counter = 0
        for i in range(epoch):
            batches = self.create_batches(x,y,batch_size)
            for batch in batches:
                batch_x = batch[0]
                batch_y = batch[1]
                prediction = self.forward_pass(batch_x)
                loss = prediction - batch_y
                cost_history.append(np.average(loss**2)/2)
                self.backward_pass(loss)
                if len(cost_history) < 2:
                    continue
                if abs((cost_history[-1]-cost_history[-2])/cost_history[-1]) < cutoff:
                    return cost_history
                if counter == iterations:
                    break
                counter += 1
        return cost_history

    def create_batches(self,x,y,batch_size):
        samples = x.shape[0]
        if batch_size > samples:
            raise Exception("Batch size cannot be larger than the dataset size.")
        indexes = np.arange(samples)
        np.random.shuffle(indexes)
        x_shuffled = x[indexes]
        y_shuffled = y[indexes]
        batches = len(x)//batch_size
        output = np.empty((batches,2,batch_size,x.shape[1]))
        for i in range(batches):
            output[i,0] = x_shuffled[i*batch_size:(i+1)*batch_size]
            output[i,1] = y_shuffled[i*batch_size:(i+1)*batch_size]
        return output

In [73]:
#Testing create_batches function
#Works as intended!!!
# (samples,features)
x = np.random.uniform(-1,1,size=(10,10))
y = np.random.uniform(-1,1,size=(10,10))
model = Model(np.empty(0))
model.create_batches(x,y,2).shape

(5, 2, 2, 10)

In [59]:
x = np.random.uniform(-1,1,size=(2,3))
print(x)
test = Layer(2,2,None,np.array([[10,3],[5,9]]),np.array([[0],[0]]))
print(test.weights)
print(test.biases)

print(test.forward_pass(x))
print(test.backward_pass(np.array([[10,3,5],[5,9,6]])))
